# GridLock — Route Resolver (refactored endpoint resolution)

This keeps the working route reconstruction logic, but fixes the endpoint lookup problem.

The important change is simple:

```text
endpoint name
    ↓
normalize case / punctuation / abbreviations
    ↓
search ALL named point sources
    ├─ cached OSM substations / switches / plants
    ├─ HIFLD substation points
    ├─ named point candidates Codex already found
    └─ manual aliases such as VCS1 → Virgil C. Summer
    ↓
use exact / containment / small spelling-variation matching
    ↓
use voltage as supporting evidence, NOT a hard rejection
    ↓
only then fall back to named line geometry
    ↓
route through the real transmission-line network
```

Examples this refactor is specifically meant to catch:

- `McIntosh` ↔ `MCINTOSH` — capitalization should never matter.
- `Winnsboro` ↔ `WINNESBORO` — small spelling differences should not block a match.
- `Faber Place` ↔ `Faber Place Substation` — a named station should not be rejected just because OSM's voltage tags are incomplete.
- `Belv Sw Sta` ↔ `Belvedere Switching Station` — common utility shorthand can have explicit aliases.
- `VCS1` / `VCS2` ↔ Virgil C. Summer naming.

Existing assets still require real public geometry. This notebook does not invent straight-line routes.


## 1. Imports and project paths

This uses the path constants you already keep under `src/gridlock`.

The only libraries beyond the Python standard library are `pandas` and `matplotlib`.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
from difflib import SequenceMatcher
import heapq
import json
import math
import re
import time

import pandas as pd
import requests

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)

from gridlock.paths import (
    PROJECT_ROOT,
    DATA_DIR,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    DOCS_DIR,
    NOTEBOOKS_DIR,
)

REVIEW_CSV = PROCESSED_DATA_DIR / "geo/dominion_asset_review.csv"

PUBLIC_GIS_ROOT = RAW_DATA_DIR / "public_gis"
OSM_ROOT = RAW_DATA_DIR / "osm"

OUTPUT_CSV = PROCESSED_DATA_DIR / "dominion/dominion_route_resolution_v3.csv"
OUTPUT_GEOJSON = PROCESSED_DATA_DIR / "geo/dominion_resolved_existing_routes_v3.geojson"

ENDPOINT_REGISTRY_CSV = (
    PROCESSED_DATA_DIR
    / "dominion/dominion_endpoint_registry.csv"
)

ENDPOINT_REGISTRY_GEOJSON = (
    PROCESSED_DATA_DIR
    / "geo/dominion_endpoint_registry.geojson"
)

for output in [
    OUTPUT_CSV,
    OUTPUT_GEOJSON,
    ENDPOINT_REGISTRY_CSV,
    ENDPOINT_REGISTRY_GEOJSON,
]:
    output.parent.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Review CSV:", REVIEW_CSV)
print("Public GIS:", PUBLIC_GIS_ROOT)
print("OSM:", OSM_ROOT)


Project root: C:\Users\sebas\PycharmProjects\Git\Project_secret_project
Review CSV: C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\processed\geo\dominion_asset_review.csv
Public GIS: C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\raw\public_gis
OSM: C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\raw\osm


## 2. Explicit routing settings

These are simple safety rails.

- `SNAP_MAX_KM` = how far a station point may be from a HIFLD line before we refuse to attach it.
- Known SCE&G / Dominion owner names are preferred.
- `NOT AVAILABLE` is treated as **unknown**, not as Dominion.
- Unknown-owner lines are considered only if a route cannot be built using known Dominion/SCE&G ownership.


In [2]:
UTILITY_OWNER_ALIASES = [
    "Dominion Energy South Carolina",
    "Dominion Energy",
    "SCE&G",
    "South Carolina Electric & Gas",
    "South Carolina Electric&Gas",
    "South Carolina Electric&Gas Company",
    "South Carolina Electric&Gas Co",
]

UNKNOWN_OWNERS = {
    "",
    "not available",
    "unknown",
    "none",
    "null",
    "nan",
}

# Neighbor-state point features are useful for explicit tie-line endpoints.
POINT_SEARCH_STATES = {"SC", "GA", "NC"}

# HIFLD public substation point layer.
HIFLD_SUBSTATIONS_QUERY = (
    "https://services.arcgis.com/HQ0xoN0EzDPBOEci/"
    "ArcGIS/rest/services/Substations/FeatureServer/0/query"
)

HIFLD_POINT_CACHE = (
    RAW_DATA_DIR
    / "public_gis"
    / "endpoint_points"
)
HIFLD_POINT_CACHE.mkdir(parents=True, exist_ok=True)

# Utility shorthand / known naming expansions.
# These are search aliases, not replacements for the original project text.
MANUAL_ENDPOINT_ALIASES = {
    "vcs1": [
        "Virgil C Summer Nuclear #1",
        "Virgil C Summer",
        "V C Summer",
    ],
    "vcs2": [
        "Virgil C Summer Nuclear #2",
        "Virgil C Summer",
        "V C Summer",
    ],
    "belv sw sta": [
        "Belvedere Switching Station",
        "Belvedere",
    ],
    "richland mall goab": [
        "Richland Mall",
    ],
    "frogmore transmission": [
        "Frogmore",
    ],
    "st helena": [
        "Saint Helena",
        "St Helena",
    ],
}

# Station/line anchor must be reasonably close to a transmission feature.
SNAP_MAX_KM = 2.0

# Keep the route finder from wandering across the regional grid.
MAX_ROUTE_KM = 400
MAX_ROUTE_HOPS = 40

# Roughly ~100 m. Used to connect small GIS endpoint digitizing differences.
SPATIAL_ROUND_DECIMALS = 3

# Small spelling differences: Winnsboro / Winnesboro, Urquhart / Uruquart, etc.
TYPO_MATCH_MIN = 0.82


## 3. Load Codex's asset review

`review` remains the full candidate table.

`assets` is a cleaner checklist containing one row per `asset_key`.

That means:

```text
review = one asset can have many candidate GIS rows
assets = one row per actual thing the project description says exists
```


In [3]:
if not REVIEW_CSV.exists():
    raise FileNotFoundError(
        f"{REVIEW_CSV} was not found. "
        "Put dominion_asset_review.csv in data/processed or change REVIEW_CSV."
    )

review = pd.read_csv(REVIEW_CSV)

required = {
    "asset_key",
    "project_key",
    "project_id",
    "project_name",
    "asset_type",
    "asset_name",
    "asset_stage",
    "endpoint_a",
    "endpoint_b",
    "voltage_kv",
    "state",
}

missing = required - set(review.columns)

if missing:
    raise ValueError(
        "dominion_asset_review.csv is missing: "
        + ", ".join(sorted(missing))
    )


def first_useful(series):
    for value in series:
        if pd.isna(value):
            continue
        if isinstance(value, str) and not value.strip():
            continue
        return value
    return None


asset_fields = [
    "project_key",
    "project_id",
    "project_name",
    "state",
    "source_page",
    "action_type",
    "detected_actions",
    "asset_type",
    "asset_name",
    "asset_identifier",
    "voltage_kv",
    "circuit",
    "endpoint_a",
    "endpoint_b",
    "asset_stage",
    "scope_notes",
    "description",
    "extracted_identifiers",
    "official_map_evidence",
    "project_coverage",
]

present_fields = [c for c in asset_fields if c in review.columns]

assets = (
    review.groupby("asset_key", as_index=False)
    .agg({column: first_useful for column in present_fields})
)

print("Candidate rows:", len(review))
print("Unique assets:", len(assets))
display(assets.head(15))


Candidate rows: 442
Unique assets: 109


,asset_key,project_key,project_id,project_name,state,source_page,action_type,detected_actions,asset_type,asset_name,asset_identifier,voltage_kv,circuit,endpoint_a,endpoint_b,asset_stage,scope_notes,description,extracted_identifiers,official_map_evidence,project_coverage
0,SC:scrtp_projects.pdf:10:5392 A-C:1,SC:scrtp_projects.pdf:10:5392 A-C,5392 A-C,Coit – Gills Creek 115kV: Construct,SC,10,construct,construct,line,Coit - Gills Creek existing 33 kV line,NaN,33,NaN,Coit,Gills Creek,existing,Convert existing line to 115 kV; approximately 1.25 miles.,Convert the existing Coit - Gills Creek 33 kV Line to 115 kV. Approximately 1.25 miles. Construct Gills Creek Terminal at Coit Bus #1.,"{""name_clues"": [""coit"", ""gills creek""], ""line_identifiers"": [], ""voltage_clues_kv"": [""115"", ""33""], ""circuit_clues"": [], ""description_asset_types"": [""line""], ""at_clauses"": [""Coi...",NaN,none
1,SC:scrtp_projects.pdf:10:5392 A-C:2,SC:scrtp_projects.pdf:10:5392 A-C,5392 A-C,Coit – Gills Creek 115kV: Construct,SC,10,construct,construct,station,Coit,NaN,115,1.0,NaN,NaN,existing,Construct Gills Creek terminal at Coit Bus #1.,Convert the existing Coit - Gills Creek 33 kV Line to 115 kV. Approximately 1.25 miles. Construct Gills Creek Terminal at Coit Bus #1.,"{""name_clues"": [""coit"", ""gills creek""], ""line_identifiers"": [], ""voltage_clues_kv"": [""115"", ""33""], ""circuit_clues"": [], ""description_asset_types"": [""line""], ""at_clauses"": [""Coi...",NaN,none
2,"SC:scrtp_projects.pdf:11:0139 M,N:1","SC:scrtp_projects.pdf:11:0139 M,N","0139 M,N","Okatie 230-115kV Substation, Jasper – Yemassee 230kV #1 Fold-in",SC,11,other,NaN,station,Okatie,NaN,230;115,NaN,NaN,NaN,existing,Expand station and add autobank.,"Expand existing Okatie Transmission sub, add a 230-115 autobank and fold the Jasper – Yemassee 230 kV #1 line into the 230 kV side. Construct two additional 115 kV bus tie lines.","{""name_clues"": [""okatie"", ""jasper"", ""yemassee""], ""line_identifiers"": [], ""voltage_clues_kv"": [""230"", ""115""], ""circuit_clues"": [""1""], ""description_asset_types"": [""line""], ""at_cl...",NaN,none
3,"SC:scrtp_projects.pdf:11:0139 M,N:2","SC:scrtp_projects.pdf:11:0139 M,N","0139 M,N","Okatie 230-115kV Substation, Jasper – Yemassee 230kV #1 Fold-in",SC,11,other,NaN,line,Jasper - Yemassee #1 fold-in,NaN,230,1.0,Jasper,Yemassee,existing,Fold into Okatie; fold-in segment limits needed.,"Expand existing Okatie Transmission sub, add a 230-115 autobank and fold the Jasper – Yemassee 230 kV #1 line into the 230 kV side. Construct two additional 115 kV bus tie lines.","{""name_clues"": [""okatie"", ""jasper"", ""yemassee""], ""line_identifiers"": [], ""voltage_clues_kv"": [""230"", ""115""], ""circuit_clues"": [""1""], ""description_asset_types"": [""line""], ""at_cl...",NaN,none
4,"SC:scrtp_projects.pdf:11:0139 M,N:3","SC:scrtp_projects.pdf:11:0139 M,N","0139 M,N","Okatie 230-115kV Substation, Jasper – Yemassee 230kV #1 Fold-in",SC,11,other,NaN,line,Okatie 115 kV bus tie 1,NaN,115,1.0,NaN,NaN,proposed,First of two additional bus ties; route needed.,"Expand existing Okatie Transmission sub, add a 230-115 autobank and fold the Jasper – Yemassee 230 kV #1 line into the 230 kV side. Construct two additional 115 kV bus tie lines.","{""name_clues"": [""okatie"", ""jasper"", ""yemassee""], ""line_identifiers"": [], ""voltage_clues_kv"": [""230"", ""115""], ""circuit_clues"": [""1""], ""description_asset_types"": [""line""], ""at_cl...",NaN,none
5,"SC:scrtp_projects.pdf:11:0139 M,N:4","SC:scrtp_projects.pdf:11:0139 M,N","0139 M,N","Okatie 230-115kV Substation, Jasper – Yemassee 230kV #1 Fold-in",SC,11,other,NaN,line,Okatie 115 kV bus tie 2,NaN,115,2.0,NaN,NaN,proposed,Second of two additional bus ties; route needed.,"Expand existing Okatie Transmission sub, add a 230-115 autobank and fold the Jasper – Yemassee 230 kV #1 line into the 230 kV side. Construct two additional 115 kV bus tie lines.","{""name_clues"": [""okatie"", ""jasper"", ""yemassee"

## 4. Text, voltage, owner, and geometry helpers

`difflib.SequenceMatcher` is built into Python.

We use it only for small spelling differences in place names. It is **not** a confidence-scoring system.

For example:

```text
Urquhart
Uruquart
```

should not fail just because OSM has a typo.

In [56]:
GENERIC_PLACE_WORDS = {
    "substation",
    "station",
    "switching",
    "switchyard",
    "transmission",
    "hydro",
    "plant",
    "facility",
    "tap",
    "junction",
    "jct",
    "line",
}


def normalize_name(value):
    """
    Case and punctuation never determine whether two facility names match.
    """
    if value is None or pd.isna(value):
        return ""

    value = str(value).lower()
    value = value.replace("&", " and ")
    value = value.replace("–", "-").replace("—", "-")

    replacements = {
        r"\bst[.]?\b": "saint",
        r"\bjct\b": "junction",
        r"\bsw\s+sta\b": "switching station",
        r"\bsta\b": "station",
        r"\bsub\b": "substation",
    }

    for pattern, replacement in replacements.items():
        value = re.sub(pattern, replacement, value)

    value = re.sub(r"[^a-z0-9#]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def core_place_name(value):
    tokens = normalize_name(value).split()

    while tokens and tokens[-1] in GENERIC_PLACE_WORDS:
        tokens.pop()

    return " ".join(tokens)


ELECTRICAL_SUFFIX_WORDS = {
    "substation",
    "station",
    "switching",
    "switchyard",
    "transmission",
    "tap",
    "junction",
}

def meaningful_tokens(value):
    return set(core_place_name(value).split())


def numeric_tokens(value):
    return re.findall(r"\d+", normalize_name(value))


def short_identity_tokens(value):
    return {
        token
        for token in normalize_name(value).split()
        if token.isalpha()
        and len(token) <= 4
        and token not in ELECTRICAL_SUFFIX_WORDS
    }


def station_match_kind(alias, feature_name):
    a = core_place_name(alias)
    b = core_place_name(feature_name)

    if not a or not b:
        return None

    # 1. Exact normalized facility name
    # McIntosh == MCINTOSH
    if a == b:
        return "exact"

    # 2. Containment is allowed only when the extra words
    # are generic electrical facility words.
    #
    # Faber Place
    # Faber Place Substation
    # -> valid
    a_tokens = set(normalize_name(alias).split())
    b_tokens = set(normalize_name(feature_name).split())

    if a_tokens <= b_tokens:
        extras = b_tokens - a_tokens

        if extras and extras <= ELECTRICAL_SUFFIX_WORDS:
            return "contains"

    if b_tokens <= a_tokens:
        extras = a_tokens - b_tokens

        if extras and extras <= ELECTRICAL_SUFFIX_WORDS:
            return "contains"

    # 3. Before fuzzy matching, identifiers must agree.

    # VCS1 must never fuzzy-match VCS2.
    nums_a = numeric_tokens(alias)
    nums_b = numeric_tokens(feature_name)

    if nums_a or nums_b:
        if nums_a != nums_b:
            return None

    # CAE must not fuzzy-match COLA.
    short_a = short_identity_tokens(alias)
    short_b = short_identity_tokens(feature_name)

    if short_a or short_b:
        if short_a != short_b:
            return None

    # 4. Small genuine spelling variations
    # Winnsboro / Winnesboro
    if min(len(a), len(b)) >= 6:
        ratio = SequenceMatcher(None, a, b).ratio()

        if ratio >= TYPO_MATCH_MIN:
            return "typo"

    return None


def place_matches(a, b):
    return station_match_kind(a, b) is not None


def parse_voltage(value):
    if value is None or pd.isna(value):
        return None

    values = re.findall(r"\d+(?:\.\d+)?", str(value))

    if not values:
        return None

    return float(values[0])


def parse_tag_voltages(value):
    """
    OSM often stores voltage in volts:
    '230000;115000;23900'
    """
    if value is None or pd.isna(value):
        return []

    result = []

    for token in re.findall(r"\d+(?:\.\d+)?", str(value)):
        number = float(token)

        if number > 1000:
            number /= 1000

        result.append(number)

    return result


def point_voltage_support(
    requested_voltage,
    voltage_values=None,
    min_voltage=None,
    max_voltage=None,
):
    """
    Voltage is evidence, not an identity filter.

    A named station is NOT discarded simply because the public voltage tag
    is incomplete or appears to conflict with the project record.
    """
    requested = parse_voltage(requested_voltage)

    if requested is None:
        return "unknown"

    values = [
        float(v)
        for v in (voltage_values or [])
        if v is not None and not pd.isna(v)
    ]

    if values:
        if any(abs(v - requested) < 0.5 for v in values):
            return "match"
        return "conflict"

    try:
        low = float(min_voltage)
    except (TypeError, ValueError):
        low = None

    try:
        high = float(max_voltage)
    except (TypeError, ValueError):
        high = None

    if low is not None and high is not None:
        if low <= requested <= high:
            return "match"
        return "conflict"

    if low is not None and abs(low - requested) < 0.5:
        return "match"

    if high is not None and abs(high - requested) < 0.5:
        return "match"

    return "unknown"


def voltage_matches(requested, candidate):
    """
    Strict line-voltage helper.

    Unlike station identity, an actual route segment should match the
    project's requested line voltage when voltage metadata is available.
    """
    requested = parse_voltage(requested)

    if requested is None:
        return True

    if candidate is None:
        return True

    if isinstance(candidate, (list, tuple, set)):
        values = candidate
    else:
        values = parse_tag_voltages(candidate)

    return any(abs(float(v) - requested) < 0.5 for v in values)


def normalize_owner(value):
    return normalize_name(value)


OWNER_ALIASES = [
    normalize_owner(x)
    for x in UTILITY_OWNER_ALIASES
]


def owner_kind(owner):
    owner_norm = normalize_owner(owner)

    if owner_norm in UNKNOWN_OWNERS:
        return "unknown"

    if any(
        alias in owner_norm or owner_norm in alias
        for alias in OWNER_ALIASES
    ):
        return "dominion"

    return "other"


def parse_json_cell(value):
    if value is None or pd.isna(value):
        return None

    if isinstance(value, (dict, list)):
        return value

    try:
        return json.loads(str(value))
    except (json.JSONDecodeError, TypeError):
        return None


def geometry_parts(geometry):
    if not geometry:
        return []

    geom_type = geometry.get("type")
    coordinates = geometry.get("coordinates") or []

    if geom_type == "LineString":
        return [coordinates]

    if geom_type == "MultiLineString":
        return coordinates

    if "paths" in geometry:
        return geometry["paths"]

    return []


def all_geometry_points(geometry):
    if not geometry:
        return []

    geom_type = geometry.get("type")
    coordinates = geometry.get("coordinates") or []

    if geom_type == "Point":
        return [tuple(coordinates[:2])]

    if geom_type == "LineString":
        return [tuple(p[:2]) for p in coordinates]

    if geom_type == "MultiLineString":
        return [
            tuple(p[:2])
            for part in coordinates
            for p in part
        ]

    if geom_type == "Polygon":
        return [
            tuple(p[:2])
            for ring in coordinates
            for p in ring
        ]

    if geom_type == "MultiPolygon":
        return [
            tuple(p[:2])
            for polygon in coordinates
            for ring in polygon
            for p in ring
        ]

    return []


def geometry_center(geometry):
    points = all_geometry_points(geometry)

    if not points:
        return None

    lon = sum(p[0] for p in points) / len(points)
    lat = sum(p[1] for p in points) / len(points)

    return lon, lat


def haversine_km(lon1, lat1, lon2, lat2):
    r = 6371.0088

    lon1, lat1, lon2, lat2 = map(
        math.radians,
        [lon1, lat1, lon2, lat2],
    )

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1)
        * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    return 2 * r * math.asin(math.sqrt(a))


def geometry_length_km(geometry):
    total = 0.0

    for part in geometry_parts(geometry):
        for first, second in zip(part, part[1:]):
            total += haversine_km(
                first[0],
                first[1],
                second[0],
                second[1],
            )

    return total


def unique_names(values):
    seen = set()
    output = []

    for value in values:
        if value is None or pd.isna(value):
            continue

        value = str(value).strip()
        key = normalize_name(value)

        if key and key not in seen:
            seen.add(key)
            output.append(value)

    return output


## 5. Read the GIS candidates Codex already found

This is now our **first** source, not something we throw away.

In particular, named OSM transmission lines can be extremely useful:

```text
Project: Urquhart - Toolebeck
OSM:     Uruquart - Toolebeck 115kV
```

If a candidate's published line name contains both project endpoints at the correct voltage, we can use its real OSM geometry immediately.

If OSM split that named line into several ways, we keep all of those geometries.

In [57]:
candidate_review = review[
    review["geometry"].notna()
    & review["matched_feature_name"].notna()
].copy()

candidate_review["geometry_obj"] = candidate_review["geometry"].apply(parse_json_cell)
candidate_review["tags_obj"] = candidate_review["source_tags"].apply(parse_json_cell)

print("Candidate rows with geometry:", len(candidate_review))
print()
display(
    candidate_review[
        [
            "asset_key",
            "asset_name",
            "matched_feature_name",
            "source",
            "match_basis",
            "asset_status",
        ]
    ].head(20)
)


Candidate rows with geometry: 371



,asset_key,asset_name,matched_feature_name,source,match_basis,asset_status
6,SC:scrtp_projects.pdf:4:6846 A:1,Eastover - Sumter DESC tie section,UNKNOWN120856 - INTERNATIONAL PAPER EASTOVER FACILITY,public_gis,one_endpoint_only,candidate
7,SC:scrtp_projects.pdf:4:6846 A:1,Eastover - Sumter DESC tie section,INTERNATIONAL PAPER EASTOVER FACILITY - NOT AVAILABLE,public_gis,one_endpoint_only,candidate
8,SC:scrtp_projects.pdf:5:6809 N:1,Batesburg - Saluda County,Batesburg - Ward 115kV,overpass,one_endpoint_only,candidate
9,SC:scrtp_projects.pdf:5:6809 N:1,Batesburg - Saluda County,Batesburg - Ward 115kV,overpass,one_endpoint_only,candidate
10,SC:scrtp_projects.pdf:5:6809 N:1,Batesburg - Saluda County,Batesburg - Ward 46kV,overpass,one_endpoint_only,candidate
12,SC:scrtp_projects.pdf:7:6852:1,Urquhart - Toolebeck,Urquhart - Fairfax 115kV,overpass,one_endpoint_only,candidate
13,SC:scrtp_projects.pdf:7:6852:1,Urquhart - Toolebeck,Uruquart - Toolebeck 115kV,overpass,one_endpoint_only,candidate
14,SC:scrtp_projects.pdf:7:6852:1,Urquhart - Toolebeck,Urquhart - Fairfax 115kV,overpass,one_endpoint_only,candidate
15,SC:scrtp_projects.pdf:7:6852:1,Urquhart - Toolebeck,Urquhart - Fairfax 115kV,overpass,one_endpoint_only,candidate
16,SC:scrtp_projects.pdf:7:6852:1,Urquhart - Toolebeck,Urquhart - Fairfax 115kV,overpass,one_endpoint_only,candidate


## 6. Endpoint aliases from the project title

Descriptions sometimes use an engineering/interconnection identifier instead of the ordinary station name.

Example:

```text
description endpoint: DESCSQ #1151
project title:         Fairfax-Yemassee
```

So the title gives us `Fairfax` as a fallback search name.

We keep the recorded endpoint; the title name is only an alias.

In [58]:
TITLE_PAIR_PATTERN = re.compile(
    r"^\s*(?P<a>.+?)\s*[-–—]\s*(?P<b>.+?)\s+"
    r"(?P<voltage>\d{2,3}(?:\s*/\s*\d{2,3})?)\s*kV\b",
    re.IGNORECASE,
)


def title_pair(project_name):
    match = TITLE_PAIR_PATTERN.search(str(project_name or ""))

    if not match:
        return None

    return (
        match.group("a").strip(" ,:-"),
        match.group("b").strip(" ,:-"),
    )


def manual_aliases(value):
    key = normalize_name(value)

    aliases = []

    for raw_key, raw_aliases in MANUAL_ENDPOINT_ALIASES.items():
        if normalize_name(raw_key) == key:
            aliases.extend(raw_aliases)

    return aliases


def endpoint_aliases(asset):
    """
    Endpoint aliases must describe THIS endpoint.

    Do not automatically assign project-title endpoints by position.
    """
    a = asset.get("endpoint_a")
    b = asset.get("endpoint_b")

    a_aliases = [a] + manual_aliases(a)
    b_aliases = [b] + manual_aliases(b)

    return (
        unique_names(a_aliases),
        unique_names(b_aliases),
    )


## 7. Parse named OSM line candidates

A candidate is considered a strong named OSM route only when:

- it is an OSM/Overpass feature,
- its name looks like `A - B ... kV`,
- its voltage agrees with the project asset,
- both sides of the published line name correspond to the project's endpoint aliases.

This is much stricter than accepting every `one_endpoint_only` candidate.

In [59]:
NAMED_LINE_PATTERN = re.compile(
    r"^\s*(?P<a>.+?)\s*[-–—]\s*(?P<b>.+?)"
    r"(?:\s+(?P<voltage>\d{2,3})\s*kV)?\s*$",
    re.IGNORECASE,
)


def candidate_voltage(row):
    tags = row.get("tags_obj") or {}

    if isinstance(tags, dict) and tags.get("voltage") is not None:
        values = parse_tag_voltages(tags.get("voltage"))
        if values:
            return values

    name = str(row.get("matched_feature_name") or "")
    values = re.findall(r"\b(\d{2,3})\s*kV\b", name, flags=re.IGNORECASE)

    return [float(x) for x in values]


def parse_named_line(name):
    match = NAMED_LINE_PATTERN.match(str(name or ""))

    if not match:
        return None

    a = match.group("a").strip(" ,:-")
    b = match.group("b").strip(" ,:-")
    voltage = match.group("voltage")

    # Strip an ending voltage if the regex left it inside endpoint B.
    b = re.sub(r"\s+\d{2,3}\s*kV$", "", b, flags=re.IGNORECASE).strip()

    return {
        "a": a,
        "b": b,
        "voltage": float(voltage) if voltage else None,
    }


def side_matches_any(side_name, aliases):
    return any(place_matches(side_name, alias) for alias in aliases)


def strong_named_osm_candidates(asset):
    rows = candidate_review[
        (candidate_review["asset_key"] == asset["asset_key"])
        & (candidate_review["source"].astype(str).str.lower() == "overpass")
    ].copy()

    if rows.empty:
        return pd.DataFrame()

    aliases_a, aliases_b = endpoint_aliases(asset)
    requested_voltage = parse_voltage(asset.get("voltage_kv"))

    accepted = []

    for idx, row in rows.iterrows():
        parsed = parse_named_line(row["matched_feature_name"])

        if not parsed:
            continue

        voltages = candidate_voltage(row)

        if requested_voltage is not None and voltages:
            if not any(abs(v - requested_voltage) < 0.5 for v in voltages):
                continue

        forward = (
            side_matches_any(parsed["a"], aliases_a)
            and side_matches_any(parsed["b"], aliases_b)
        )

        reverse = (
            side_matches_any(parsed["a"], aliases_b)
            and side_matches_any(parsed["b"], aliases_a)
        )

        if forward or reverse:
            accepted.append(idx)

    if not accepted:
        return pd.DataFrame()

    return rows.loc[accepted].copy()


def combine_candidate_geometries(rows):
    parts = []

    for geometry in rows["geometry_obj"]:
        parts.extend(geometry_parts(geometry))

    if not parts:
        return None

    return {
        "type": "MultiLineString",
        "coordinates": parts,
    }


## 8. Load the cached HIFLD/DOE line network

The raw line chunks Codex downloaded are reopened locally.

We do **not** assume that a file stored under `/SC/` proves every line inside it is physically in South Carolina. The folder is treated as the download region, not as feature-level truth.

Ownership and spatial anchors will constrain the route.

In [60]:
def arcgis_geometry_to_geojson(geometry):
    if not geometry:
        return None

    if "type" in geometry and "coordinates" in geometry:
        return geometry

    paths = geometry.get("paths")

    if not paths:
        return None

    if len(paths) == 1:
        return {
            "type": "LineString",
            "coordinates": paths[0],
        }

    return {
        "type": "MultiLineString",
        "coordinates": paths,
    }


def lower_keys(mapping):
    return {
        str(key).lower(): value
        for key, value in (mapping or {}).items()
    }


def load_netl_file(path):
    data = json.loads(path.read_text(encoding="utf-8"))

    if isinstance(data, dict):
        features = data.get("features", [])
    elif isinstance(data, list):
        features = data
    else:
        features = []

    rows = []

    for feature in features:
        attrs = lower_keys(
            feature.get("properties")
            or feature.get("attributes")
            or {}
        )

        geometry = arcgis_geometry_to_geojson(
            feature.get("geometry")
        )

        if not geometry:
            continue

        voltage = attrs.get("voltage")

        try:
            voltage = float(voltage)
        except (TypeError, ValueError):
            voltage = None

        rows.append({
            "feature_id": (
                attrs.get("id")
                or attrs.get("objectid")
                or attrs.get("objectid_1")
                or feature.get("id")
            ),
            "sub_1": attrs.get("sub_1"),
            "sub_2": attrs.get("sub_2"),
            "owner": attrs.get("owner"),
            "owner_kind": owner_kind(attrs.get("owner")),
            "voltage": voltage,
            "status": attrs.get("status"),
            "source_detail": attrs.get("source"),
            "geometry": geometry,
            "length_km": geometry_length_km(geometry),
            "source_file": str(path),
        })

    return rows


netl_files = sorted(
    PUBLIC_GIS_ROOT.glob("*/netl_lines_*.json")
)

if not netl_files:
    raise FileNotFoundError(
        f"No netl_lines_*.json files found below {PUBLIC_GIS_ROOT}"
    )

line_rows = []

for path in netl_files:
    loaded = load_netl_file(path)
    line_rows.extend(loaded)
    print(
        path.relative_to(PROJECT_ROOT),
        "->",
        f"{len(loaded):,}",
        "features",
    )

lines = pd.DataFrame(line_rows)

lines["_dedupe"] = (
    lines["feature_id"].astype(str)
    + "|"
    + lines["voltage"].astype(str)
)

lines = (
    lines
    .drop_duplicates("_dedupe")
    .drop(columns="_dedupe")
    .reset_index(drop=True)
)

# Treat the sentinel as missing.
lines.loc[lines["voltage"] < 0, "voltage"] = None

print()
print("Unique transmission features:", len(lines))
display(lines.head())


data\raw\public_gis\SC\netl_lines_0.json -> 2,000 features
data\raw\public_gis\SC\netl_lines_2000.json -> 2,000 features
data\raw\public_gis\SC\netl_lines_4000.json -> 793 features

Unique transmission features: 4793


,feature_id,sub_1,sub_2,owner,owner_kind,voltage,status,source_detail,geometry,length_km,source_file
0,141156,UNKNOWN120901,TAP150391,NOT AVAILABLE,unknown,NaN,IN SERVICE,IMAGERY,"{'type': 'LineString', 'coordinates': [[-80.5960297937522, 34.2735811826351], [-80.5980022452828, 34.273019167862486], [-80.59844958293807, 34.27314765927612]]}",0.235232,C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\raw\public_gis\SC\netl_lines_0.json
1,140829,UNKNOWN114377,TAP148337,PROGRESS ENERGY CAROLINAS INC,other,230.0,IN SERVICE,"OpenStreetMap, EIA 861, EIA 860","{'type': 'LineString', 'coordinates': [[-79.72569149980578, 34.960807500284595], [-79.7259806998351, 34.96072069971552], [-79.72829210009543, 34.95975059980982], [-79.729748100...",0.983018,C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\raw\public_gis\SC\netl_lines_0.json
2,140830,UNKNOWN114019,TAP154445,"DUKE ENERGY CAROLINAS, LLC",other,100.0,IN SERVICE,"IMAGERY, EIA 861, EIA 860","{'type': 'LineString', 'coordinates': [[-82.28903053294879, 34.82895261015645], [-82.28914284302056, 34.8287736752237], [-82.28924944340037, 34.828609967843406], [-82.290707577...",2.079769,C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\raw\public_gis\SC\netl_lines_0.json
3,140904,UNKNOWN115977,TAP152614,"DUKE ENERGY CAROLINAS, LLC",other,NaN,IN SERVICE,IMAGERY,"{'type': 'LineString', 'coordinates': [[-81.48966593591474, 35.0328962629465], [-81.48591388317875, 35.02681886784264], [-81.48056009566362, 35.022128951042745], [-81.474770867...",9.495848,C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\raw\public_gis\SC\netl_lines_0.json
4,140843,UNKNOWN142327,TAP150545,DUKE ENERGY PROGRESS - (NC),other,115.0,IN SERVICE,"IMAGERY, EIA 860","{'type': 'LineString', 'coordinates': [[-80.27341360727779, 33.59702573017798], [-80.27320381831959, 33.597306903248665]]}",0.036811,C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\raw\public_gis\SC\netl_lines_0.json


## 9. Load all cached named OSM power-point features

The previous version only looked at the cached `power_substations_overpass.json`.

This refactor scans the cached OSM JSON files and keeps named:

- substations
- switches
- plants / switchyards

Line ways are ignored here because this section is only building endpoint **points**.

Voltage tags are preserved as evidence but do not decide whether a named station exists.


In [61]:
def overpass_element_geometry(element):
    if "lat" in element and "lon" in element:
        return {
            "type": "Point",
            "coordinates": [
                float(element["lon"]),
                float(element["lat"]),
            ],
        }

    geometry = element.get("geometry")

    if geometry and isinstance(geometry, list):
        coords = [
            [float(p["lon"]), float(p["lat"])]
            for p in geometry
            if "lon" in p and "lat" in p
        ]

        if len(coords) >= 3 and coords[0] == coords[-1]:
            return {
                "type": "Polygon",
                "coordinates": [coords],
            }

        if coords:
            return {
                "type": "LineString",
                "coordinates": coords,
            }

    center = element.get("center")

    if center:
        return {
            "type": "Point",
            "coordinates": [
                float(center["lon"]),
                float(center["lat"]),
            ],
        }

    return None


OSM_POINT_POWER_TYPES = {
    "substation",
    "switch",
    "plant",
}


def load_named_osm_power_points(path, state):
    try:
        data = json.loads(
            path.read_text(encoding="utf-8")
        )
    except Exception:
        return []

    rows = []

    for element in data.get("elements", []):
        tags = element.get("tags") or {}
        power_type = tags.get("power")
        name = tags.get("name")

        if power_type not in OSM_POINT_POWER_TYPES:
            continue

        if not name:
            continue

        geometry = overpass_element_geometry(element)
        center = geometry_center(geometry)

        if not geometry or not center:
            continue

        rows.append({
            "state": state,
            "feature_id": f"{element.get('type')}/{element.get('id')}",
            "name": name,
            "operator": tags.get("operator"),
            "operator_kind": owner_kind(tags.get("operator")),
            "voltage_values": parse_tag_voltages(tags.get("voltage")),
            "min_voltage": None,
            "max_voltage": None,
            "feature_type": power_type,
            "geometry": geometry,
            "lon": center[0],
            "lat": center[1],
            "point_source": "osm",
            "source_file": str(path),
        })

    return rows


osm_point_rows = []

for state_dir in sorted(OSM_ROOT.glob("*")):
    if not state_dir.is_dir():
        continue

    state = state_dir.name.upper()

    for path in sorted(state_dir.glob("*.json")):
        loaded = load_named_osm_power_points(
            path,
            state,
        )

        if loaded:
            osm_point_rows.extend(loaded)


osm_points = pd.DataFrame(osm_point_rows)

if not osm_points.empty:
    osm_points = (
        osm_points
        .drop_duplicates(
            ["state", "feature_id"]
        )
        .reset_index(drop=True)
    )

print("Named cached OSM power points:", len(osm_points))

if not osm_points.empty:
    display(
        osm_points[
            [
                "state",
                "name",
                "feature_type",
                "operator",
                "voltage_values",
                "point_source",
            ]
        ].head(20)
    )


Named cached OSM power points: 561


,state,name,feature_type,operator,voltage_values,point_source
0,SC,Wylie Dam,plant,Duke Energy,[],osm
1,SC,Central Energy Facilities,plant,Clemson University - Main Campus,[],osm
2,SC,Keowee Power Plant,plant,Duke Energy,[],osm
3,SC,Catawba Nuclear Station,plant,Duke Energy,[],osm
4,SC,Hilton Head Power Plant,plant,South Carolina Public Service Authority,[],osm
5,SC,Jocassee Hydroelectric Station,plant,Duke Energy,[],osm
6,SC,McMeekin Station,plant,"Dominion Energy South Carolina, Inc",[],osm
7,SC,Saluda Hydroelectric Plant,plant,Dominion Energy South Carolina,[],osm
8,SC,W.S. Lee Power Plant,plant,Duke Energy,[],osm
9,SC,Wateree Dam Hydro,plant,Duke Energy,[],osm


## 10. Load HIFLD substation points

This was the biggest missing source in the old route resolver.

The old notebook loaded HIFLD **lines**, but not the HIFLD **substation point layer**. That is why an exact public feature such as `MCINTOSH` could exist without ever being considered as an endpoint.

The query is cached locally. After the first successful download, later runs reuse the local GeoJSON.


In [62]:
def lower_keys(mapping):
    return {
        str(key).lower(): value
        for key, value in (mapping or {}).items()
    }


def fetch_hifld_substations(state):
    cache_path = (
        HIFLD_POINT_CACHE
        / f"hifld_substations_{state}.geojson"
    )

    if cache_path.exists():
        return json.loads(
            cache_path.read_text(encoding="utf-8")
        )

    session = requests.Session()
    session.headers.update({
        "User-Agent": (
            "GridLock-Hackathon/"
            "endpoint-resolution"
        )
    })

    features = []
    offset = 0
    page_size = 2000

    while True:
        response = session.get(
            HIFLD_SUBSTATIONS_QUERY,
            params={
                "where": f"STATE = '{state}'",
                "outFields": "*",
                "returnGeometry": "true",
                "outSR": 4326,
                "f": "geojson",
                "resultOffset": offset,
                "resultRecordCount": page_size,
            },
            timeout=120,
        )
        response.raise_for_status()

        data = response.json()
        batch = data.get("features", [])
        features.extend(batch)

        if len(batch) < page_size:
            break

        offset += len(batch)
        time.sleep(0.2)

    collection = {
        "type": "FeatureCollection",
        "features": features,
    }

    cache_path.write_text(
        json.dumps(collection),
        encoding="utf-8",
    )

    return collection


hifld_point_rows = []

for state in sorted(POINT_SEARCH_STATES):
    try:
        collection = fetch_hifld_substations(
            state
        )
    except Exception as exc:
        print(
            f"HIFLD point download failed for {state}:",
            exc,
        )
        continue

    for feature in collection.get(
        "features",
        [],
    ):
        props = lower_keys(
            feature.get("properties") or {}
        )
        geometry = feature.get("geometry") or {}

        if geometry.get("type") != "Point":
            continue

        coords = geometry.get("coordinates") or []

        if len(coords) < 2:
            continue

        name = (
            props.get("name")
            or props.get("sub_name")
            or props.get("station")
        )

        # UNKNOWN/TAP names are still retained in the source table,
        # but only genuinely matching names become endpoint anchors.
        hifld_point_rows.append({
            "state": (
                props.get("state")
                or state
            ),
            "feature_id": (
                props.get("id")
                or props.get("objectid")
                or props.get("objectid_1")
            ),
            "name": name,
            "operator": (
                props.get("owner")
                or props.get("operator")
            ),
            "operator_kind": owner_kind(
                props.get("owner")
                or props.get("operator")
            ),
            "voltage_values": [],
            "min_voltage": props.get("min_volt"),
            "max_voltage": props.get("max_volt"),
            "feature_type": (
                props.get("type")
                or "substation"
            ),
            "geometry": geometry,
            "lon": float(coords[0]),
            "lat": float(coords[1]),
            "point_source": "hifld_substation",
            "source_file": (
                HIFLD_SUBSTATIONS_QUERY
            ),
        })


hifld_points = pd.DataFrame(
    hifld_point_rows
)

print(
    "HIFLD substation points:",
    len(hifld_points),
)

if not hifld_points.empty:
    display(
        hifld_points[
            [
                "state",
                "name",
                "min_voltage",
                "max_voltage",
                "operator",
            ]
        ].head(20)
    )


HIFLD substation points: 8186


,state,name,min_voltage,max_voltage,operator
0,GA,UNKNOWN107666,115,230,None
1,GA,UNKNOWN107689,69,115,None
2,GA,MORGANVILLE,-999999,-999999,None
3,GA,UNKNOWN109648,-999999,-999999,None
4,GA,UNKNOWN109649,-999999,-999999,None
5,GA,UNKNOWN109680,-999999,230,None
6,GA,UNKNOWN109681,-999999,-999999,None
7,GA,WALLACEVILLE,-999999,161,None
8,GA,UNKNOWN109791,-999999,-999999,None
9,GA,YOUNG RW,230,230,None


## 11. Reuse named point candidates Codex already discovered

`dominion_asset_review.csv` already contains some useful named OSM candidates.

Instead of making those discoveries asset-specific and throwing them away, this cell turns them into a reusable point-feature table that any endpoint can match against.


In [63]:
review_point_rows = []

for _, row in candidate_review.iterrows():
    geometry = row.get("geometry_obj")
    center = geometry_center(geometry)

    if not center:
        continue

    tags = row.get("tags_obj") or {}
    power_type = (
        tags.get("power")
        if isinstance(tags, dict)
        else None
    )

    match_basis = str(
        row.get("match_basis") or ""
    ).lower()

    # Keep actual named point/facility candidates,
    # not generic transmission-line geometry.
    is_point_candidate = (
        power_type in {
            "substation",
            "switch",
            "plant",
        }
        or "station_name" in match_basis
    )

    if not is_point_candidate:
        continue

    name = row.get("matched_feature_name")

    if name is None or pd.isna(name):
        continue

    review_point_rows.append({
        "state": row.get("state"),
        "feature_id": row.get("feature_id"),
        "name": name,
        "operator": (
            tags.get("operator")
            if isinstance(tags, dict)
            else None
        ),
        "operator_kind": owner_kind(
            tags.get("operator")
            if isinstance(tags, dict)
            else None
        ),
        "voltage_values": parse_tag_voltages(
            tags.get("voltage")
            if isinstance(tags, dict)
            else None
        ),
        "min_voltage": None,
        "max_voltage": None,
        "feature_type": (
            power_type
            or "candidate_station"
        ),
        "geometry": geometry,
        "lon": center[0],
        "lat": center[1],
        "point_source": "review_named_candidate",
        "source_file": row.get("source_file"),
    })


review_points = pd.DataFrame(
    review_point_rows
)

point_tables = [
    table
    for table in [
        osm_points,
        hifld_points,
        review_points,
    ]
    if not table.empty
]

if point_tables:
    point_features = pd.concat(
        point_tables,
        ignore_index=True,
    )
else:
    point_features = pd.DataFrame(
        columns=[
            "state",
            "feature_id",
            "name",
            "operator",
            "operator_kind",
            "voltage_values",
            "min_voltage",
            "max_voltage",
            "feature_type",
            "geometry",
            "lon",
            "lat",
            "point_source",
            "source_file",
        ]
    )


print(
    "Total reusable named point features:",
    len(point_features),
)

display(
    point_features[
        [
            "point_source",
            "state",
            "name",
            "feature_type",
            "operator",
        ]
    ].head(25)
)


Total reusable named point features: 8764


,point_source,state,name,feature_type,operator
0,osm,SC,Wylie Dam,plant,Duke Energy
1,osm,SC,Central Energy Facilities,plant,Clemson University - Main Campus
2,osm,SC,Keowee Power Plant,plant,Duke Energy
3,osm,SC,Catawba Nuclear Station,plant,Duke Energy
4,osm,SC,Hilton Head Power Plant,plant,South Carolina Public Service Authority
5,osm,SC,Jocassee Hydroelectric Station,plant,Duke Energy
6,osm,SC,McMeekin Station,plant,"Dominion Energy South Carolina, Inc"
7,osm,SC,Saluda Hydroelectric Plant,plant,Dominion Energy South Carolina
8,osm,SC,W.S. Lee Power Plant,plant,Duke Energy
9,osm,SC,Wateree Dam Hydro,plant,Duke Energy


## 10. Also turn named candidate lines into endpoint anchors

Sometimes OSM has a named line even when the station itself is not named.

Example:

```text
Graniteville - Ward 230kV
```

That line can still tell us approximately where the `Graniteville` side is.

So each strongly named candidate line can contribute its first/last geometry point as an endpoint anchor.

In [64]:
def geometry_terminal_points(geometry):
    terminals = []

    for part in geometry_parts(geometry):
        if len(part) >= 2:
            terminals.append(tuple(part[0][:2]))
            terminals.append(tuple(part[-1][:2]))

    return terminals


def candidate_line_endpoint_anchors(asset, aliases):
    """
    Build endpoint anchors from named OSM line candidate geometries.

    If a candidate line says A-B, its first/last geometry points are possible
    anchors for A/B. We keep both orientations because OSM geometry direction
    does not have to match the written name order.
    """
    rows = candidate_review[
        (candidate_review["asset_key"] == asset["asset_key"])
        & (candidate_review["source"].astype(str).str.lower() == "overpass")
    ]

    anchors = []

    for _, row in rows.iterrows():
        parsed = parse_named_line(row["matched_feature_name"])

        if not parsed:
            continue

        requested_voltage = parse_voltage(asset.get("voltage_kv"))
        voltages = candidate_voltage(row)

        if requested_voltage is not None and voltages:
            if not any(abs(v - requested_voltage) < 0.5 for v in voltages):
                continue

        geometry = row["geometry_obj"]
        terminals = geometry_terminal_points(geometry)

        if len(terminals) < 2:
            continue

        first = terminals[0]
        last = terminals[-1]

        # If the line's first named side matches this endpoint alias,
        # either geometry end could still be the actual named side.
        if side_matches_any(parsed["a"], aliases):
            anchors.extend([
                {
                    "lon": first[0],
                    "lat": first[1],
                    "anchor_name": parsed["a"],
                    "anchor_source": "named_osm_line_terminal",
                    "feature_id": row["feature_id"],
                },
                {
                    "lon": last[0],
                    "lat": last[1],
                    "anchor_name": parsed["a"],
                    "anchor_source": "named_osm_line_terminal",
                    "feature_id": row["feature_id"],
                },
            ])

        if side_matches_any(parsed["b"], aliases):
            anchors.extend([
                {
                    "lon": first[0],
                    "lat": first[1],
                    "anchor_name": parsed["b"],
                    "anchor_source": "named_osm_line_terminal",
                    "feature_id": row["feature_id"],
                },
                {
                    "lon": last[0],
                    "lat": last[1],
                    "anchor_name": parsed["b"],
                    "anchor_source": "named_osm_line_terminal",
                    "feature_id": row["feature_id"],
                },
            ])

    return anchors


## 12. Resolve endpoint names against every point source

This replaces the old `station_anchors()` logic.

Key differences:

- capitalization is ignored;
- small spelling differences are accepted;
- manual shorthand aliases are searched;
- HIFLD substations are searched;
- Georgia / neighboring-state point features are available for tie endpoints;
- voltage helps rank a match but **never deletes a strong name match**;
- operator helps rank a match but does not automatically reject an intertie endpoint.

This is where `McIntosh → MCINTOSH`, `Winnsboro → WINNESBORO`, and `Faber Place → Faber Place Substation` are supposed to be caught.


In [65]:
NAME_RANK = {
    "exact": 0,
    "contains": 1,
    "typo": 2,
}

VOLTAGE_RANK = {
    "match": 0,
    "unknown": 1,
    "conflict": 2,
}

OWNER_RANK = {
    "dominion": 0,
    "unknown": 1,
    "other": 2,
}

SOURCE_RANK = {
    "review_named_candidate": 0,
    "osm": 1,
    "hifld_substation": 1,
    "hifld_line_name_cluster": 2,
    "named_osm_line_terminal": 3,
}


def best_alias_match(
    aliases,
    feature_name,
):
    matches = []

    for alias in aliases:
        kind = station_match_kind(
            alias,
            feature_name,
        )

        if kind:
            matches.append(
                (
                    NAME_RANK[kind],
                    kind,
                    alias,
                )
            )

    if not matches:
        return None

    _, kind, alias = min(matches)

    return {
        "name_match": kind,
        "matched_alias": alias,
    }


def named_point_anchors(
    asset,
    aliases,
):
    if point_features.empty:
        return []

    requested_voltage = parse_voltage(
        asset.get("voltage_kv")
    )

    asset_state = str(
        asset.get("state") or ""
    ).upper()

    found = []

    for _, feature in point_features.iterrows():
        feature_name = feature.get("name")

        if feature_name is None or pd.isna(feature_name):
            continue

        match = best_alias_match(
            aliases,
            feature_name,
        )

        if not match:
            continue

        feature_state = str(
            feature.get("state") or ""
        ).upper()

        voltage_support = point_voltage_support(
            requested_voltage,
            voltage_values=(
                feature.get("voltage_values")
                if isinstance(
                    feature.get("voltage_values"),
                    list,
                )
                else []
            ),
            min_voltage=feature.get(
                "min_voltage"
            ),
            max_voltage=feature.get(
                "max_voltage"
            ),
        )

        anchor = {
            "lon": float(feature["lon"]),
            "lat": float(feature["lat"]),
            "anchor_name": feature_name,
            "anchor_source": feature.get(
                "point_source"
            ),
            "feature_id": feature.get(
                "feature_id"
            ),
            "feature_state": feature_state,
            "feature_type": feature.get(
                "feature_type"
            ),
            "name_match": match["name_match"],
            "matched_alias": match["matched_alias"],
            "voltage_support": voltage_support,
            "operator": feature.get(
                "operator"
            ),
            "operator_kind": feature.get(
                "operator_kind"
            ),
            "source_file": feature.get(
                "source_file"
            ),
        }

        anchor["_sort_key"] = (
            NAME_RANK.get(
                anchor["name_match"],
                9,
            ),
            0
            if feature_state == asset_state
            else 1,
            SOURCE_RANK.get(
                anchor["anchor_source"],
                9,
            ),
            VOLTAGE_RANK.get(
                voltage_support,
                9,
            ),
            OWNER_RANK.get(
                anchor["operator_kind"],
                9,
            ),
        )

        found.append(anchor)

    found.sort(
        key=lambda row: row["_sort_key"]
    )

    # Keep several plausible same-name candidates so the physical
    # transmission network can disambiguate genuinely repeated names.
    for anchor in found:
        anchor.pop("_sort_key", None)

    return found[:8]


## 13. Fallback: derive a point from repeated HIFLD line endpoint names

Some HIFLD substations are named `UNKNOWN...` even when HIFLD transmission lines around the same location use the real station name.

If no named point feature exists, this fallback looks for several same-voltage HIFLD lines that mention the endpoint name. Their physical line terminals usually cluster around the real station.

A single named line is not enough to infer which end is the station, so this fallback requires a repeated terminal cluster.


In [66]:
def geometry_terminal_points(geometry):
    terminals = []

    for part in geometry_parts(geometry):
        if len(part) >= 2:
            terminals.append(
                tuple(part[0][:2])
            )
            terminals.append(
                tuple(part[-1][:2])
            )

    return terminals


def hifld_line_name_cluster_anchors(
    asset,
    aliases,
):
    requested_voltage = parse_voltage(
        asset.get("voltage_kv")
    )

    if requested_voltage is None:
        return []

    subset = lines[
        lines["voltage"].notna()
        & lines["voltage"]
        .sub(requested_voltage)
        .abs()
        .lt(0.5)
    ]

    terminal_rows = []

    for idx, line in subset.iterrows():
        names = [
            line.get("sub_1"),
            line.get("sub_2"),
        ]

        matches = [
            best_alias_match(
                aliases,
                name,
            )
            for name in names
            if name is not None
            and not pd.isna(name)
        ]

        matches = [
            match
            for match in matches
            if match
        ]

        if not matches:
            continue

        best_match = min(
            matches,
            key=lambda match: NAME_RANK[
                match["name_match"]
            ],
        )

        for lon, lat in geometry_terminal_points(
            line["geometry"]
        ):
            terminal_rows.append({
                "lon": lon,
                "lat": lat,
                "line_index": idx,
                "feature_id": line.get(
                    "feature_id"
                ),
                "name_match": best_match[
                    "name_match"
                ],
                "matched_alias": best_match[
                    "matched_alias"
                ],
            })

    if not terminal_rows:
        return []

    # True station terminals should repeat at nearly the same location
    # across several incident HIFLD features. Opposite line ends scatter.
    buckets = defaultdict(list)

    for row in terminal_rows:
        key = (
            round(
                row["lon"],
                SPATIAL_ROUND_DECIMALS,
            ),
            round(
                row["lat"],
                SPATIAL_ROUND_DECIMALS,
            ),
        )
        buckets[key].append(row)

    clusters = sorted(
        buckets.items(),
        key=lambda item: len(item[1]),
        reverse=True,
    )

    anchors = []

    for _, members in clusters:
        distinct_features = {
            member["feature_id"]
            for member in members
        }

        if len(distinct_features) < 2:
            continue

        lon = sum(
            member["lon"]
            for member in members
        ) / len(members)

        lat = sum(
            member["lat"]
            for member in members
        ) / len(members)

        best = min(
            members,
            key=lambda member: NAME_RANK[
                member["name_match"]
            ],
        )

        anchors.append({
            "lon": lon,
            "lat": lat,
            "anchor_name": (
                best["matched_alias"]
                + " [HIFLD line-name cluster]"
            ),
            "anchor_source": "hifld_line_name_cluster",
            "feature_id": json.dumps(
                sorted(
                    str(x)
                    for x in distinct_features
                )
            ),
            "feature_state": str(
                asset.get("state") or ""
            ).upper(),
            "feature_type": "line_terminal_cluster",
            "name_match": best["name_match"],
            "matched_alias": best["matched_alias"],
            "voltage_support": "match",
            "operator": None,
            "operator_kind": "unknown",
            "source_file": "cached HIFLD line network",
        })

    return anchors[:3]


## 14. Build endpoint anchors once and reuse them

This is now a reusable endpoint resolver.

Search order:

1. named point features from OSM / HIFLD / Codex review;
2. repeated HIFLD line-name terminal clusters;
3. named OSM line terminals as a last fallback.

Results are cached by endpoint aliases + voltage + state, so the same endpoint is not rediscovered for every project row.


In [67]:
ENDPOINT_ANCHOR_CACHE = {}


def build_endpoint_anchors(
    asset,
    side,
):
    aliases_a, aliases_b = endpoint_aliases(
        asset
    )

    aliases = (
        aliases_a
        if side == "a"
        else aliases_b
    )

    cache_key = (
        tuple(
            sorted(
                normalize_name(alias)
                for alias in aliases
            )
        ),
        parse_voltage(
            asset.get("voltage_kv")
        ),
        str(
            asset.get("state") or ""
        ).upper(),
    )

    if cache_key in ENDPOINT_ANCHOR_CACHE:
        return [
            dict(anchor)
            for anchor in ENDPOINT_ANCHOR_CACHE[
                cache_key
            ]
        ]

    # 1. Real named point features.
    anchors = named_point_anchors(
        asset,
        aliases,
    )

    # 2. If no named point exists, infer the physical node from
    #    repeated same-name HIFLD line terminals.
    if not anchors:
        anchors = (
            hifld_line_name_cluster_anchors(
                asset,
                aliases,
            )
        )

    # 3. Last fallback: named OSM line terminals.
    if not anchors:
        anchors = (
            candidate_line_endpoint_anchors(
                asset,
                aliases,
            )
        )

    seen = set()
    unique = []

    for anchor in anchors:
        key = (
            round(anchor["lon"], 5),
            round(anchor["lat"], 5),
            anchor.get("anchor_source"),
            str(anchor.get("feature_id")),
        )

        if key in seen:
            continue

        seen.add(key)
        unique.append(anchor)

    ENDPOINT_ANCHOR_CACHE[
        cache_key
    ] = [
        dict(anchor)
        for anchor in unique
    ]

    return unique


## 15. Endpoint registry — inspect the no-brain matches before routing

This table is intentionally shown **before** the pathfinder.

It lets us immediately see things like:

```text
McIntosh  → MCINTOSH       → exact → HIFLD
Winnsboro → WINNESBORO     → typo  → HIFLD
Faber Place → Faber Place Substation → exact/contains → OSM
```

`voltage_support="conflict"` does not delete a named facility. It is just a warning to inspect the public voltage metadata.


In [68]:
endpoint_registry_rows = []

existing_line_assets = assets[
    (
        assets["asset_type"]
        .astype(str)
        .str.lower()
        == "line"
    )
    &
    (
        assets["asset_stage"]
        .astype(str)
        .str.lower()
        == "existing"
    )
].copy()


for _, asset in existing_line_assets.iterrows():
    for side, column in [
        ("a", "endpoint_a"),
        ("b", "endpoint_b"),
    ]:
        endpoint_name = asset.get(column)

        if (
            endpoint_name is None
            or pd.isna(endpoint_name)
            or not str(endpoint_name).strip()
        ):
            continue

        anchors = build_endpoint_anchors(
            asset,
            side,
        )

        best = (
            anchors[0]
            if anchors
            else {}
        )

        endpoint_registry_rows.append({
            "endpoint_name": endpoint_name,
            "normalized_endpoint": normalize_name(
                endpoint_name
            ),
            "state": asset.get("state"),
            "voltage_kv": parse_voltage(
                asset.get("voltage_kv")
            ),
            "found": bool(anchors),
            "resolved_name": best.get(
                "anchor_name"
            ),
            "longitude": best.get("lon"),
            "latitude": best.get("lat"),
            "source": best.get(
                "anchor_source"
            ),
            "feature_id": best.get(
                "feature_id"
            ),
            "feature_type": best.get(
                "feature_type"
            ),
            "name_match": best.get(
                "name_match"
            ),
            "matched_alias": best.get(
                "matched_alias"
            ),
            "voltage_support": best.get(
                "voltage_support"
            ),
            "operator": best.get(
                "operator"
            ),
            "operator_kind": best.get(
                "operator_kind"
            ),
            "source_file": best.get(
                "source_file"
            ),
        })


endpoint_registry = (
    pd.DataFrame(
        endpoint_registry_rows
    )
    .sort_values(
        [
            "normalized_endpoint",
            "voltage_kv",
            "found",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
    .drop_duplicates(
        [
            "normalized_endpoint",
            "voltage_kv",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)


print("Unique endpoint/voltage pairs:", len(endpoint_registry))

display(
    endpoint_registry[
        "found"
    ]
    .value_counts(dropna=False)
    .to_frame("endpoint_voltage_pairs")
)

display(
    endpoint_registry[
        [
            "endpoint_name",
            "voltage_kv",
            "found",
            "resolved_name",
            "source",
            "name_match",
            "matched_alias",
            "voltage_support",
            "operator",
        ]
    ]
)


Unique endpoint/voltage pairs: 84


,endpoint_voltage_pairs
found,
True,45
False,39


,endpoint_name,voltage_kv,found,resolved_name,source,name_match,matched_alias,voltage_support,operator
0,Adams Run,46.0,True,Adams Run Substation,osm,exact,Adams Run,conflict,NaN
1,Aiken PSA,46.0,False,NaN,NaN,NaN,NaN,NaN,NaN
2,Ashley River,115.0,False,NaN,NaN,NaN,NaN,NaN,NaN
3,Batesburg,115.0,True,Batesburg Substation,osm,exact,Batesburg,match,NaN
4,Bayfront,115.0,False,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
79,Wateree,230.0,True,Wateree Substation,osm,exact,Wateree,match,NaN
80,Williams,230.0,True,Williams Substation,osm,exact,Williams,match,NaN
81,Winnsboro,115.0,True,WINNESBORO,hifld_substation,typo,Winnsboro,match,NaN
82,Yemassee,115.0,True,Yemassee Substation,review_named_candidate,exact,Yemassee,conflict,South Carolina Electric & Gas Company


## 13. HIFLD route network for one voltage

We no longer have a hard-coded `69 kV` cutoff.

Instead:

```text
Need 115 kV?
→ search 115 kV if the downloaded data contains it.

Need 66 kV?
→ search 66 kV if it exists.

Need 46 kV?
→ if HIFLD has no 46 kV lines, HIFLD simply cannot solve that route.
→ OSM named lines may still solve it.
```

Known Dominion/SCE&G owner features are tried first.

If that fails, we make a second attempt that also allows `NOT AVAILABLE` owner features.

In [69]:
def voltage_network(voltage, allow_unknown_owner=False):
    voltage = parse_voltage(voltage)

    if voltage is None:
        return pd.DataFrame()

    subset = lines[
        lines["voltage"].notna()
        & lines["voltage"].sub(voltage).abs().lt(0.5)
    ].copy()

    allowed = {"dominion"}

    if allow_unknown_owner:
        allowed.add("unknown")

    subset = subset[
        subset["owner_kind"].isin(allowed)
    ].copy()

    return subset


## 14. Snap a real endpoint point to nearby HIFLD line geometry

The station name no longer has to appear in `SUB_1` or `SUB_2`.

We simply ask:

> Which same-voltage transmission features physically pass closest to this real station point?

The closest few lines within `SNAP_MAX_KM` are retained as possible network anchors.

In [70]:
def point_to_geometry_vertex_km(lon, lat, geometry):
    points = all_geometry_points(geometry)

    if not points:
        return float("inf")

    return min(
        haversine_km(
            lon,
            lat,
            point[0],
            point[1],
        )
        for point in points
    )


def snap_anchors_to_lines(
    anchors,
    network,
    max_km=SNAP_MAX_KM,
):
    snapped = []

    if network.empty:
        return snapped

    for anchor in anchors:
        distances = []

        for idx, line in network.iterrows():
            distance = point_to_geometry_vertex_km(
                anchor["lon"],
                anchor["lat"],
                line["geometry"],
            )

            if distance <= max_km:
                distances.append(
                    (
                        distance,
                        idx,
                    )
                )

        distances.sort()

        # Keep at most 5 nearby line candidates per point.
        for distance, idx in distances[:5]:
            snapped.append({
                **anchor,
                "line_index": idx,
                "snap_distance_km": distance,
            })

    return snapped


## 15. Build a spatial graph from actual HIFLD line geometry

The old resolver mostly walked `SUB_1` / `SUB_2` names.

This one walks the **geometry itself**.

Each published line part becomes an edge between its two physical terminal coordinates.

Nearby endpoints are rounded to about 100 m so tiny GIS digitizing differences do not break an otherwise continuous corridor.

In [71]:
def snap_node(point):
    return (
        round(float(point[0]), SPATIAL_ROUND_DECIMALS),
        round(float(point[1]), SPATIAL_ROUND_DECIMALS),
    )


def build_spatial_graph(network):
    graph = defaultdict(list)
    line_nodes = defaultdict(set)

    for idx, line in network.iterrows():
        for part in geometry_parts(line["geometry"]):
            if len(part) < 2:
                continue

            node_a = snap_node(part[0])
            node_b = snap_node(part[-1])

            if node_a == node_b:
                continue

            edge = {
                "line_index": idx,
                "length_km": float(line["length_km"]),
            }

            graph[node_a].append(
                (node_b, edge)
            )

            graph[node_b].append(
                (node_a, edge)
            )

            line_nodes[idx].add(node_a)
            line_nodes[idx].add(node_b)

    return graph, line_nodes


## 16. Find the shortest real-line route between snapped endpoint lines

This is ordinary Dijkstra shortest-path logic.

The important thing is what the graph represents:

```text
node = physical endpoint of a published HIFLD line feature
edge = actual HIFLD transmission-line geometry
```

The path result preserves every feature used.

In [72]:
def shortest_spatial_route(
    network,
    start_snaps,
    end_snaps,
):
    if not start_snaps or not end_snaps:
        return None

    graph, line_nodes = build_spatial_graph(network)

    end_line_indexes = {
        item["line_index"]
        for item in end_snaps
    }

    target_nodes = set()

    for idx in end_line_indexes:
        target_nodes.update(
            line_nodes.get(idx, set())
        )

    heap = []
    best = {}

    for item in start_snaps:
        idx = item["line_index"]

        if idx not in network.index:
            continue

        feature_length = float(
            network.loc[idx, "length_km"]
        )

        for node in line_nodes.get(idx, set()):
            heapq.heappush(
                heap,
                (
                    feature_length,
                    1,
                    node,
                    [idx],
                ),
            )

            best[(node, 1)] = feature_length

    while heap:
        distance, hops, node, path = heapq.heappop(heap)

        if distance > MAX_ROUTE_KM:
            continue

        if hops > MAX_ROUTE_HOPS:
            continue

        if node in target_nodes:
            # If an end feature is already in the route, we are done.
            if any(
                idx in end_line_indexes
                for idx in path
            ):
                return {
                    "line_indexes": list(dict.fromkeys(path)),
                    "route_length_km": distance,
                    "route_hops": hops,
                }

            # Otherwise attach the shortest target feature touching this node.
            target_features = [
                idx
                for idx in end_line_indexes
                if node in line_nodes.get(idx, set())
            ]

            if target_features:
                target_idx = min(
                    target_features,
                    key=lambda idx: float(
                        network.loc[idx, "length_km"]
                    ),
                )

                final_path = path + [target_idx]

                return {
                    "line_indexes": list(dict.fromkeys(final_path)),
                    "route_length_km": (
                        distance
                        + float(
                            network.loc[
                                target_idx,
                                "length_km",
                            ]
                        )
                    ),
                    "route_hops": hops + 1,
                }

        for neighbor, edge in graph.get(node, []):
            new_distance = (
                distance
                + edge["length_km"]
            )

            new_hops = hops + 1

            if (
                new_distance > MAX_ROUTE_KM
                or new_hops > MAX_ROUTE_HOPS
            ):
                continue

            state = (
                neighbor,
                new_hops,
            )

            if new_distance >= best.get(
                state,
                float("inf"),
            ):
                continue

            best[state] = new_distance

            heapq.heappush(
                heap,
                (
                    new_distance,
                    new_hops,
                    neighbor,
                    path + [edge["line_index"]],
                ),
            )

    return None


## 17. Direct HIFLD endpoint-name match remains available

We no longer depend on it, but when HIFLD actually does name both ends correctly, that is useful evidence.

This checks for a direct same-voltage feature whose `SUB_1` and `SUB_2` correspond to the project aliases.

In [73]:
def direct_hifld_match(asset, network):
    aliases_a, aliases_b = endpoint_aliases(asset)

    matches = []

    for idx, line in network.iterrows():
        sub_1 = line.get("sub_1")
        sub_2 = line.get("sub_2")

        if not sub_1 or not sub_2:
            continue

        forward = (
            side_matches_any(sub_1, aliases_a)
            and side_matches_any(sub_2, aliases_b)
        )

        reverse = (
            side_matches_any(sub_1, aliases_b)
            and side_matches_any(sub_2, aliases_a)
        )

        if forward or reverse:
            matches.append(idx)

    return matches


## 18. Convert selected line features into one route geometry

A project corridor may consist of many real line features.

We preserve them as a `MultiLineString` and keep every feature ID in the output.

In [74]:
def route_geometry_from_indexes(
    network,
    line_indexes,
):
    parts = []

    for idx in line_indexes:
        parts.extend(
            geometry_parts(
                network.loc[idx, "geometry"]
            )
        )

    if not parts:
        return None

    return {
        "type": "MultiLineString",
        "coordinates": parts,
    }


def route_segment_summary(
    network,
    line_indexes,
):
    unique_indexes = list(
        dict.fromkeys(line_indexes)
    )

    rows = network.loc[unique_indexes]

    return {
        "segment_count": len(rows),
        "segment_feature_ids": json.dumps(
            rows["feature_id"]
            .astype(str)
            .tolist()
        ),
        "segment_names": json.dumps(
            [
                f"{row['sub_1']} - {row['sub_2']}"
                for _, row in rows.iterrows()
            ]
        ),
        "segment_owners": json.dumps(
            sorted(
                {
                    str(owner)
                    for owner in rows["owner"].dropna()
                    if str(owner).strip()
                }
            )
        ),
        "source_files": json.dumps(
            sorted(
                set(
                    rows["source_file"]
                    .astype(str)
                )
            )
        ),
    }


## 19. Resolve one existing line asset

Resolution order:

```text
A. Strong named OSM route already found by Codex
B. Direct HIFLD endpoint-to-endpoint feature
C. OSM station/name anchors → known-owner HIFLD spatial route
D. Same anchors → HIFLD route allowing unknown-owner gaps
E. unresolved
```

If unknown-owner segments are required, the route is returned as `review_required`, not silently promoted to fully resolved.

In [75]:
def base_result(asset):
    return {
        "asset_key": asset["asset_key"],
        "project_key": asset.get("project_key"),
        "project_id": asset.get("project_id"),
        "project_name": asset.get("project_name"),
        "state": asset.get("state"),
        "action_type": asset.get("action_type"),
        "asset_type": asset.get("asset_type"),
        "asset_name": asset.get("asset_name"),
        "asset_stage": asset.get("asset_stage"),
        "endpoint_a": asset.get("endpoint_a"),
        "endpoint_b": asset.get("endpoint_b"),
        "voltage_kv": asset.get("voltage_kv"),
        "official_map_evidence": asset.get("official_map_evidence"),
        "route_status": None,
        "route_method": None,
        "route_length_km": None,
        "route_hops": None,
        "segment_count": 0,
        "segment_feature_ids": None,
        "segment_names": None,
        "segment_owners": None,
        "source_files": None,
        "endpoint_a_anchor": None,
        "endpoint_b_anchor": None,
        "endpoint_a_snap_km": None,
        "endpoint_b_snap_km": None,
        "geometry": None,
        "review_note": None,
    }


def resolve_existing_line(asset):
    result = base_result(asset)

    if str(asset.get("asset_type", "")).lower() != "line":
        result["route_status"] = "not_line_asset"
        return result

    if str(asset.get("asset_stage", "")).lower() != "existing":
        result["route_status"] = "future_or_proposed"
        result["review_note"] = (
            "This notebook does not force future lines into the current network."
        )
        return result

    voltage = parse_voltage(
        asset.get("voltage_kv")
    )

    if voltage is None:
        result["route_status"] = "missing_voltage"
        return result

    # ------------------------------------------------------------
    # A. Strong named OSM route already found by Codex.
    # ------------------------------------------------------------
    named_osm = strong_named_osm_candidates(asset)

    if not named_osm.empty:
        # Group all OSM ways carrying the same normalized published line name.
        name_groups = {}

        for _, row in named_osm.iterrows():
            key = normalize_name(
                row["matched_feature_name"]
            )
            name_groups.setdefault(
                key,
                [],
            ).append(row.name)

        # Prefer the group with the most geometry pieces.
        chosen_key = max(
            name_groups,
            key=lambda key: len(name_groups[key]),
        )

        chosen = named_osm.loc[
            name_groups[chosen_key]
        ]

        geometry = combine_candidate_geometries(
            chosen
        )

        if geometry:
            result["route_status"] = "resolved"
            result["route_method"] = "named_osm_line"
            result["route_length_km"] = geometry_length_km(
                geometry
            )
            result["route_hops"] = len(chosen)
            result["segment_count"] = len(chosen)
            result["segment_feature_ids"] = json.dumps(
                chosen["feature_id"]
                .astype(str)
                .tolist()
            )
            result["segment_names"] = json.dumps(
                chosen["matched_feature_name"]
                .astype(str)
                .tolist()
            )
            result["source_files"] = json.dumps(
                sorted(
                    set(
                        chosen["source_file"]
                        .dropna()
                        .astype(str)
                    )
                )
            )
            result["geometry"] = geometry
            result["review_note"] = (
                "Both project endpoints matched the published OSM line name at the correct voltage."
            )
            return result

    # ------------------------------------------------------------
    # HIFLD must actually contain the requested voltage.
    # ------------------------------------------------------------
    known_network = voltage_network(
        voltage,
        allow_unknown_owner=False,
    )

    if known_network.empty:
        result["route_status"] = "voltage_not_in_dominion_hifld"
        result["review_note"] = (
            f"No known-owner Dominion/SCE&G HIFLD lines were found at {voltage:g} kV. "
            "Use OSM or official project documentation."
        )
        return result

    # ------------------------------------------------------------
    # B. Direct HIFLD endpoint-name feature.
    # ------------------------------------------------------------
    direct = direct_hifld_match(
        asset,
        known_network,
    )

    if direct:
        # Keep all direct parallel features instead of pretending there is only one.
        summary = route_segment_summary(
            known_network,
            direct,
        )

        result.update(summary)
        result["route_status"] = "resolved"
        result["route_method"] = "direct_hifld_named_feature"
        result["route_length_km"] = sum(
            float(
                known_network.loc[
                    idx,
                    "length_km",
                ]
            )
            for idx in direct
        )
        result["route_hops"] = len(direct)
        result["geometry"] = route_geometry_from_indexes(
            known_network,
            direct,
        )

        if len(direct) > 1:
            result["review_note"] = (
                "Several same-voltage HIFLD features directly match the same endpoints; "
                "all were preserved because parallel circuits may be real."
            )

        return result

    # ------------------------------------------------------------
    # C/D. Point anchors -> spatial HIFLD route.
    # ------------------------------------------------------------
    endpoint_a_anchors = build_endpoint_anchors(
        asset,
        "a",
    )

    endpoint_b_anchors = build_endpoint_anchors(
        asset,
        "b",
    )

    if not endpoint_a_anchors or not endpoint_b_anchors:
        result["route_status"] = "endpoint_anchor_missing"
        result["review_note"] = (
            f"Endpoint anchors found: A={len(endpoint_a_anchors)}, "
            f"B={len(endpoint_b_anchors)}. "
            "Need another public location source for the missing endpoint."
        )
        return result

    # First attempt: known Dominion/SCE&G owner only.
    start_snaps = snap_anchors_to_lines(
        endpoint_a_anchors,
        known_network,
    )

    end_snaps = snap_anchors_to_lines(
        endpoint_b_anchors,
        known_network,
    )

    path = shortest_spatial_route(
        known_network,
        start_snaps,
        end_snaps,
    )

    if path:
        chosen = path["line_indexes"]
        summary = route_segment_summary(
            known_network,
            chosen,
        )

        result.update(summary)
        result["route_status"] = "resolved"
        result["route_method"] = "station_anchored_hifld_path"
        result["route_length_km"] = path["route_length_km"]
        result["route_hops"] = path["route_hops"]
        result["geometry"] = route_geometry_from_indexes(
            known_network,
            chosen,
        )

        best_a = min(
            start_snaps,
            key=lambda x: x["snap_distance_km"],
        )

        best_b = min(
            end_snaps,
            key=lambda x: x["snap_distance_km"],
        )

        result["endpoint_a_anchor"] = (
            f"{best_a['anchor_name']} [{best_a['anchor_source']}]"
        )
        result["endpoint_b_anchor"] = (
            f"{best_b['anchor_name']} [{best_b['anchor_source']}]"
        )
        result["endpoint_a_snap_km"] = best_a["snap_distance_km"]
        result["endpoint_b_snap_km"] = best_b["snap_distance_km"]

        return result

    # Second attempt: allow unknown-owner gaps, but never another named utility.
    broad_network = voltage_network(
        voltage,
        allow_unknown_owner=True,
    )

    start_snaps = snap_anchors_to_lines(
        endpoint_a_anchors,
        broad_network,
    )

    end_snaps = snap_anchors_to_lines(
        endpoint_b_anchors,
        broad_network,
    )

    path = shortest_spatial_route(
        broad_network,
        start_snaps,
        end_snaps,
    )

    if path:
        chosen = path["line_indexes"]
        summary = route_segment_summary(
            broad_network,
            chosen,
        )

        result.update(summary)
        result["route_status"] = "review_required"
        result["route_method"] = "station_anchored_hifld_path_with_unknown_owner"
        result["route_length_km"] = path["route_length_km"]
        result["route_hops"] = path["route_hops"]
        result["geometry"] = route_geometry_from_indexes(
            broad_network,
            chosen,
        )

        best_a = min(
            start_snaps,
            key=lambda x: x["snap_distance_km"],
        )

        best_b = min(
            end_snaps,
            key=lambda x: x["snap_distance_km"],
        )

        result["endpoint_a_anchor"] = (
            f"{best_a['anchor_name']} [{best_a['anchor_source']}]"
        )
        result["endpoint_b_anchor"] = (
            f"{best_b['anchor_name']} [{best_b['anchor_source']}]"
        )
        result["endpoint_a_snap_km"] = best_a["snap_distance_km"]
        result["endpoint_b_snap_km"] = best_b["snap_distance_km"]

        result["review_note"] = (
            "A route exists only when HIFLD segments with unknown ownership are allowed. "
            "Inspect the path before accepting it."
        )

        return result

    result["route_status"] = "unresolved"
    result["review_note"] = (
        "Both endpoint locations were found, but no same-voltage public HIFLD path "
        "connected their nearby Dominion/unknown-owner transmission features."
    )

    return result


## 20. Run the resolver

This is the main result table.

The first thing to inspect is whether `resolved` increases significantly and whether the remaining failures are now mostly:

- genuinely missing endpoint locations,
- future/proposed infrastructure,
- lower-voltage assets better represented in OSM,
- or routes requiring manual review.

In [76]:
route_results = pd.DataFrame(
    resolve_existing_line(row)
    for _, row in assets.iterrows()
)

print("Route status:")
display(
    route_results[
        "route_status"
    ]
    .value_counts(dropna=False)
    .to_frame("assets")
)

print("\nRoute method:")
display(
    route_results[
        "route_method"
    ]
    .value_counts(dropna=False)
    .to_frame("assets")
)


Route status:


,assets
route_status,
endpoint_anchor_missing,35
not_line_asset,30
future_or_proposed,19
voltage_not_in_dominion_hifld,11
resolved,10
unresolved,3
review_required,1



Route method:


,assets
route_method,
NaN,98
station_anchored_hifld_path,7
direct_hifld_named_feature,2
station_anchored_hifld_path_with_unknown_owner,1
named_osm_line,1


In [50]:
missing_anchor_rows = []

for _, asset in assets.iterrows():
    result = route_results[
        route_results["asset_key"]
        == asset["asset_key"]
    ]

    if result.empty:
        continue

    if (
        result.iloc[0]["route_status"]
        != "endpoint_anchor_missing"
    ):
        continue

    a_anchors = build_endpoint_anchors(
        asset,
        "a",
    )
    b_anchors = build_endpoint_anchors(
        asset,
        "b",
    )

    missing_anchor_rows.append({
        "project_name": asset["project_name"],
        "asset_name": asset["asset_name"],
        "endpoint_a": asset["endpoint_a"],
        "a_found": len(a_anchors) > 0,
        "a_match": (
            a_anchors[0].get("anchor_name")
            if a_anchors
            else None
        ),
        "a_source": (
            a_anchors[0].get("anchor_source")
            if a_anchors
            else None
        ),
        "endpoint_b": asset["endpoint_b"],
        "b_found": len(b_anchors) > 0,
        "b_match": (
            b_anchors[0].get("anchor_name")
            if b_anchors
            else None
        ),
        "b_source": (
            b_anchors[0].get("anchor_source")
            if b_anchors
            else None
        ),
        "voltage_kv": asset["voltage_kv"],
    })


missing_anchors = pd.DataFrame(
    missing_anchor_rows
)

display(missing_anchors)


,project_name,asset_name,endpoint_a,a_found,a_match,a_source,endpoint_b,b_found,b_match,b_source,voltage_kv
0,Flat Rock 115kV Sub and Tap: Construct,Belv Sw Sta - Graniteville #1,Belv Sw Sta,False,NaN,NaN,Graniteville,False,NaN,NaN,115
1,Saluda Hydro – Bush River 115kV #1 and #2 Tie Lines Rebuild,Saluda Hydro - Bush River #1,Saluda Hydro,True,Saluda Substation,osm,Bush River,False,NaN,NaN,115
2,Saluda Hydro – Bush River 115kV #1 and #2 Tie Lines Rebuild,Saluda Hydro - Bush River #2,Saluda Hydro,True,Saluda Substation,osm,Bush River,False,NaN,NaN,115
3,Yemassee– Ritter 230kV #1 & #2: Construct SPDC with B-1272,Ritter - Yemassee existing 230 kV,Ritter,False,NaN,NaN,Yemassee,True,Yemassee Substation,review_named_candidate,230
4,Fairfax-Yemassee 115kV: Upgrade for DESCSQ #1151 Interconnection,DESCSQ #1151 - Yemassee,DESCSQ #1151,False,NaN,NaN,Yemassee,True,Yemassee Substation,review_named_candidate,115
5,St George - Sumter 230kV Tie: Rebuild Line from Santee Substation - Duke/Progress Energy Tie,Santee - Duke/Progress Energy tie,Santee,True,Santee Substation,osm,Duke/Progress Energy Tie,False,NaN,NaN,230
6,Summerville 115kV Loop: Rebuild,Summerville loop wood-pole sections,NaN,False,NaN,NaN,NaN,False,NaN,NaN,115
7,Hooks - Modoc 115/46 kV Rebuild,Hooks - Modoc 115 kV,Hooks,False,NaN,NaN,Modoc,True,Modoc Substation,osm,115
8,Burton-St Helena 115kV: Rebuild Frogmore Distribution – St. Helena Section,Frogmore Transmission - St Helena,Frogmore Transmission,True,Frogmore Substation,osm,St Helena,False,NaN,NaN,115
9,Wateree-Hopkins 230kV Line #1: Rebuild,Wateree - Hopkins #1,Wateree,True,Wateree Substation,osm,Hopkins,False,NaN,NaN,230


## 21. Inspect resolved routes

`segment_count > 1` is expected.

It means several real transmission features collectively form the project's existing corridor.

In [77]:
resolved = route_results[
    route_results["route_status"].isin(
        [
            "resolved",
            "review_required",
        ]
    )
][
    [
        "project_name",
        "asset_name",
        "endpoint_a",
        "endpoint_b",
        "voltage_kv",
        "route_status",
        "route_method",
        "segment_count",
        "route_length_km",
        "endpoint_a_anchor",
        "endpoint_b_anchor",
        "endpoint_a_snap_km",
        "endpoint_b_snap_km",
        "segment_names",
        "review_note",
    ]
].copy()

display(
    resolved.sort_values(
        [
            "route_status",
            "route_method",
            "project_name",
        ]
    )
)


,project_name,asset_name,endpoint_a,endpoint_b,voltage_kv,route_status,route_method,segment_count,route_length_km,endpoint_a_anchor,endpoint_b_anchor,endpoint_a_snap_km,endpoint_b_snap_km,segment_names,review_note
3,"Okatie 230-115kV Substation, Jasper – Yemassee 230kV #1 Fold-in",Jasper - Yemassee #1 fold-in,Jasper,Yemassee,230,resolved,direct_hifld_named_feature,1,61.532893,NaN,NaN,NaN,NaN,"[""JASPER - YEMASSEE""]",NaN
56,VCS1-Denny Terrace 230kV & VCS1-Pineland 230kV: Rebuild Single Circuit Sections,VCS1 - Denny Terrace single-circuit sections,VCS1,Denny Terrace,230,resolved,direct_hifld_named_feature,1,42.461612,NaN,NaN,NaN,NaN,"[""V C SUMMER - DENNY TERRACE""]",NaN
99,Urquhart – Toolebeck 115kV line: Rebuild,Urquhart - Toolebeck,Urquhart,Toolebeck,115,resolved,named_osm_line,1,27.414648,NaN,NaN,NaN,NaN,"[""Uruquart - Toolebeck 115kV""]",Both project endpoints matched the published OSM line name at the correct voltage.
105,Cainhoy - Hamlin 115kV: Rebuild Line and Cainhoy – Hamlin 115 kV #2: Construct New 115 kV Line,Cainhoy - Hamlin existing line,Cainhoy,Hamlin,115,resolved,station_anchored_hifld_path,2,24.935858,Cainhoy Substation [review_named_candidate],Hamlin Substation [review_named_candidate],2.793145e-01,0.017598,"[""UNKNOWN120971 - TAP147729"", ""UNKNOWN121118 - TAP147729""]",NaN
16,Church Creek – Faber Place – Charleston Transmission: Add 230kV Line,Church Creek - Faber Place 230 kV,Church Creek,Faber Place,230,resolved,station_anchored_hifld_path,1,6.394187,Church Creek Substation [review_named_candidate],Faber Place Substation [review_named_candidate],3.431280e-02,0.026185,"[""UNKNOWN120998 - UNKNOWN121120""]",NaN
17,Church Creek – Faber Place – Charleston Transmission: Add 230kV Line,Church Creek - Faber Place 115 kV,Church Creek,Faber Place,115,resolved,station_anchored_hifld_path,2,10.708396,Church Creek Substation [review_named_candidate],Faber Place Substation [review_named_candidate],3.431280e-02,0.026185,"[""UNKNOWN120998 - TAP142085"", ""UNKNOWN120998 - UNKNOWN121120""]",NaN
101,Dawson 230kV Sub and Fold-in: Construct and Rebuild,Canadys - Church Creek fold-in,Canadys,Church Creek,230,resolved,station_anchored_hifld_path,2,107.894976,CANADYS [hifld_substation],Church Creek Substation [review_named_candidate],3.536620e-12,0.034313,"[""CANADYS - YEMASSEE"", ""UNKNOWN120998 - CANADYS""]",NaN
102,Dawson 230kV Sub and Fold-in: Construct and Rebuild,Canadys - Faber Place fold-in,Canadys,Faber Place,230,resolved,station_anchored_hifld_path,2,69.121316,CANADYS [hifld_substation],Faber Place Substation [review_named_candidate],3.536620e-12,0.026185,"[""UNKNOWN120969 - CANADYS"", ""UNKNOWN120969 - UNKNOWN121120""]",NaN
74,St. George- Holly Hill 115kV: Rebuild Structures 83-108,St George - Holly Hill structures 83-108,St George,Holly Hill,115,resolved,station_anchored_hifld_path,2,23.620039,Saint George Switching Station [osm],Holly Hill Substation [osm],8.458422e-02,0.014301,"[""UNKNOWN122843 - TAP142321"", ""UNKNOWN123114 - TAP142321""]",NaN
96,VCS2-Ward 230kV: Rebuild Line,VCS2 - Ward,VCS2,Ward,230,resolved,station_anchored_hifld_path,12,390.170035,V C Summer [HIFLD line-name cluster] [hifld_line_name_cluster],Ward Substation [osm],0.000000e+00,0.011189,"[""V C SUMMER - UNKNOWN117343"", ""UNKNOWN117343 - UNKNOWN117539"", ""UNKNOWN117532 - UNKNOWN117539"", ""UNKNOWN117532 - TAP142264"", ""TAP142264 - WATEREE STATION"", ""WATEREE STATION - ...",NaN


In [53]:
display(
    endpoint_registry[
        endpoint_registry["found"]
    ][
        [
            "endpoint_name",
            "voltage_kv",
            "resolved_name",
            "source",
            "name_match",
            "matched_alias",
            "voltage_support",
            "operator",
        ]
    ]
)

,endpoint_name,voltage_kv,resolved_name,source,name_match,matched_alias,voltage_support,operator
0,Adams Run,46.0,Adams Run Substation,osm,exact,Adams Run,conflict,NaN
3,Batesburg,115.0,Batesburg Substation,osm,exact,Batesburg,match,NaN
8,CAE Industrial Park,115.0,Cola Industrial Park Substation,osm,typo,CAE Industrial Park,match,NaN
9,Cainhoy,115.0,Cainhoy Substation,review_named_candidate,exact,Cainhoy,match,Dominion Energy South Carolina
11,Cameron,46.0,Cameron Substation,osm,exact,Cameron,match,NaN
12,Cameron Jct,46.0,Cameron Substation,osm,exact,Cameron Jct,match,NaN
14,Canadys,115.0,Canadys Substation,osm,exact,Canadys,match,South Carolina Electric & Gas Company
15,Canadys,230.0,Canadys Substation,osm,exact,Canadys,match,South Carolina Electric & Gas Company
16,Church Creek,115.0,Church Creek Substation,review_named_candidate,exact,Church Creek,match,NaN
17,Church Creek,230.0,Church Creek Substation,review_named_candidate,exact,Church Creek,match,NaN


In [78]:
display(
    endpoint_registry[
        ~endpoint_registry["found"]
    ][
        [
            "endpoint_name",
            "voltage_kv",
            "state",
        ]
    ]
)

,endpoint_name,voltage_kv,state
1,Aiken PSA,46.0,SC
2,Ashley River,115.0,SC
4,Bayfront,115.0,SC
5,Belv Sw Sta,115.0,SC
6,Boone Hill,115.0,SC
7,Bush River,115.0,SC
8,CAE Industrial Park,115.0,SC
10,Calhoun County,46.0,SC
13,Camp Road,115.0,SC
18,Coit,33.0,SC


## 22. Inspect what remains unresolved

This is now the useful research queue.

`endpoint_anchor_missing` means the endpoint name could not yet be turned into a real point.

`voltage_not_in_dominion_hifld` means HIFLD is not the right source at that voltage — but OSM or Dominion maps may still contain the route.

`unresolved` means both endpoints were located, but the public HIFLD line network still did not produce a defensible path.

In [79]:
needs_work = route_results[
    ~route_results["route_status"].isin(
        [
            "resolved",
            "not_line_asset",
            "future_or_proposed",
        ]
    )
][
    [
        "asset_key",
        "project_name",
        "asset_name",
        "endpoint_a",
        "endpoint_b",
        "voltage_kv",
        "route_status",
        "route_method",
        "official_map_evidence",
        "review_note",
    ]
].copy()

display(needs_work)


,asset_key,project_name,asset_name,endpoint_a,endpoint_b,voltage_kv,route_status,route_method,official_map_evidence,review_note
0,SC:scrtp_projects.pdf:10:5392 A-C:1,Coit – Gills Creek 115kV: Construct,Coit - Gills Creek existing 33 kV line,Coit,Gills Creek,33,voltage_not_in_dominion_hifld,NaN,NaN,No known-owner Dominion/SCE&G HIFLD lines were found at 33 kV. Use OSM or official project documentation.
12,SC:scrtp_projects.pdf:14:6381:3,Flat Rock 115kV Sub and Tap: Construct,Belv Sw Sta - Graniteville #1,Belv Sw Sta,Graniteville,115,endpoint_anchor_missing,NaN,NaN,"Endpoint anchors found: A=0, B=0. Need another public location source for the missing endpoint."
13,SC:scrtp_projects.pdf:15:6870 A:1,Saluda Hydro – Bush River 115kV #1 and #2 Tie Lines Rebuild,Saluda Hydro - Bush River #1,Saluda Hydro,Bush River,115,endpoint_anchor_missing,NaN,"[{""title"": ""Saluda-Bush River"", ""purpose"": ""Dominion Energy and its contractors will replace 15 miles of 115-kilovolt (kV) steel lattice tower structures with self-supporting g...","Endpoint anchors found: A=1, B=0. Need another public location source for the missing endpoint."
14,SC:scrtp_projects.pdf:15:6870 A:2,Saluda Hydro – Bush River 115kV #1 and #2 Tie Lines Rebuild,Saluda Hydro - Bush River #2,Saluda Hydro,Bush River,115,endpoint_anchor_missing,NaN,"[{""title"": ""Saluda-Bush River"", ""purpose"": ""Dominion Energy and its contractors will replace 15 miles of 115-kilovolt (kV) steel lattice tower structures with self-supporting g...","Endpoint anchors found: A=1, B=0. Need another public location source for the missing endpoint."
24,SC:scrtp_projects.pdf:17:6359:3,Yemassee– Ritter 230kV #1 & #2: Construct SPDC with B-1272,Ritter - Yemassee existing 230 kV,Ritter,Yemassee,230,endpoint_anchor_missing,NaN,"[{""title"": ""Ritter-Yemassee"", ""purpose"": ""Create awareness of the project and educate property owners on why this project needed, and to provide project details"", ""url"": ""https...","Endpoint anchors found: A=0, B=3. Need another public location source for the missing endpoint."
27,SC:scrtp_projects.pdf:18:6238 H:1,Fairfax-Yemassee 115kV: Upgrade for DESCSQ #1151 Interconnection,DESCSQ #1151 - Yemassee,DESCSQ #1151,Yemassee,115,endpoint_anchor_missing,NaN,NaN,"Endpoint anchors found: A=0, B=3. Need another public location source for the missing endpoint."
28,SC:scrtp_projects.pdf:19:6809 M:1,St George - Sumter 230kV Tie: Rebuild Line from Santee Substation - Duke/Progress Energy Tie,Santee - Duke/Progress Energy tie,Santee,Duke/Progress Energy Tie,230,endpoint_anchor_missing,NaN,NaN,"Endpoint anchors found: A=1, B=0. Need another public location source for the missing endpoint."
29,SC:scrtp_projects.pdf:1:06810 H:1,Summerville 115kV Loop: Rebuild,Summerville loop wood-pole sections,NaN,NaN,115,endpoint_anchor_missing,NaN,"[{""title"": ""Summerville Loop"", ""purpose"": ""Dominion Energy has identified several transmission lines in Dorchester County in need of replacement. Replacing the existing poles a...","Endpoint anchors found: A=0, B=0. Need another public location source for the missing endpoint."
30,SC:scrtp_projects.pdf:20:6809 G:1,Hooks - Modoc 115/46 kV Rebuild,Hooks - Modoc 115 kV,Hooks,Modoc,115,endpoint_anchor_missing,NaN,"[{""title"": ""Hooks-Modoc"", ""purpose"": ""As Dominion Energy remains focused on reliability and resiliency, we have identified two transmission lines in McCormick County near U.S. ...","Endpoint anchors found: A=0, B=1. Need another public location source for the missing endpoint."
31,SC:scrtp_projects.pdf:20:6809 G:2,Hooks - Modoc 115/46 kV Rebuild,Hooks - Modoc 46 kV,Hooks,Modoc,46,voltage_not_in_dominion_hifld,NaN,"[{""title"": ""Hooks-Modoc"", ""purpose"": ""As Dominion Energy remains focused on reliability and resiliency, we have identified two transmission lines in McCormick County near U.S. ...",No known-owner Dominion/SCE&G HIFLD lines were found at 46 kV. Use OSM or official project documentation.


## 23. Preview any resolved route

This is just a sanity check so you can visually inspect the reconstructed corridor before using it in GridLock.

In [ ]:
import matplotlib.pyplot as plt

resolved_keys = route_results.loc[
    route_results["route_status"].isin(
        [
            "resolved",
            "review_required",
        ]
    ),
    "asset_key",
].tolist()

ASSET_TO_PLOT = (
    resolved_keys[0]
    if resolved_keys
    else None
)


def plot_route(asset_key):
    row = route_results[
        route_results["asset_key"] == asset_key
    ]

    if row.empty:
        raise KeyError(asset_key)

    row = row.iloc[0]
    geometry = row["geometry"]

    if not isinstance(geometry, dict):
        print("No geometry:", asset_key)
        return

    fig, ax = plt.subplots(
        figsize=(9, 7)
    )

    for part in geometry_parts(geometry):
        xs = [p[0] for p in part]
        ys = [p[1] for p in part]

        ax.plot(
            xs,
            ys,
        )

    ax.set_title(
        f"{row['project_name']}\n"
        f"{row['asset_name']} — {row['route_method']}"
    )

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_aspect(
        "equal",
        adjustable="datalim",
    )

    plt.show()


if ASSET_TO_PLOT:
    plot_route(ASSET_TO_PLOT)
else:
    print("No resolved routes yet.")


## 24. Project-level coverage

One located line does **not** make a multi-asset project complete.

This table tells us how many existing line assets were resolved for each project.

In [ ]:
existing_lines = route_results[
    (
        route_results["asset_type"]
        .astype(str)
        .str.lower()
        == "line"
    )
    &
    (
        route_results["asset_stage"]
        .astype(str)
        .str.lower()
        == "existing"
    )
].copy()

existing_lines["is_resolved"] = (
    existing_lines["route_status"]
    .eq("resolved")
)

existing_lines["needs_review"] = (
    existing_lines["route_status"]
    .eq("review_required")
)

project_coverage = (
    existing_lines
    .groupby(
        [
            "project_key",
            "project_id",
            "project_name",
        ],
        dropna=False,
    )
    .agg(
        existing_line_assets=(
            "asset_key",
            "nunique",
        ),
        resolved_line_assets=(
            "is_resolved",
            "sum",
        ),
        review_line_assets=(
            "needs_review",
            "sum",
        ),
    )
    .reset_index()
)


def coverage_label(row):
    total = row[
        "existing_line_assets"
    ]

    solved = row[
        "resolved_line_assets"
    ]

    if total > 0 and solved == total:
        return "all existing lines resolved"

    if solved > 0:
        return "some existing lines resolved"

    return "no existing lines resolved"


project_coverage[
    "coverage"
] = project_coverage.apply(
    coverage_label,
    axis=1,
)

display(
    project_coverage[
        "coverage"
    ]
    .value_counts()
    .to_frame("projects")
)

display(project_coverage)


## 25. Save two outputs

Only two files:

```text
dominion_route_resolution_v2.csv
dominion_resolved_existing_routes_v2.geojson
```

The CSV is the audit/review table.

The GeoJSON contains only routes that were either resolved or explicitly marked `review_required`.

In [ ]:
# -------------------------
# Route audit CSV
# -------------------------
csv_out = route_results.copy()

csv_out["geometry"] = csv_out[
    "geometry"
].apply(
    lambda value: (
        json.dumps(
            value,
            separators=(",", ":"),
        )
        if isinstance(value, dict)
        else None
    )
)

csv_out.to_csv(
    OUTPUT_CSV,
    index=False,
)


# -------------------------
# Resolved route GeoJSON
# -------------------------
features = []

for _, row in route_results.iterrows():
    if row["route_status"] not in {
        "resolved",
        "review_required",
    }:
        continue

    geometry = row["geometry"]

    if not isinstance(geometry, dict):
        continue

    properties = {
        key: row.get(key)
        for key in [
            "asset_key",
            "project_key",
            "project_id",
            "project_name",
            "action_type",
            "asset_name",
            "asset_stage",
            "endpoint_a",
            "endpoint_b",
            "voltage_kv",
            "route_status",
            "route_method",
            "route_length_km",
            "route_hops",
            "segment_count",
            "segment_feature_ids",
            "segment_names",
            "segment_owners",
            "endpoint_a_anchor",
            "endpoint_b_anchor",
            "endpoint_a_snap_km",
            "endpoint_b_snap_km",
            "review_note",
        ]
    }

    for key, value in list(
        properties.items()
    ):
        if pd.isna(value):
            properties[key] = None

    features.append({
        "type": "Feature",
        "properties": properties,
        "geometry": geometry,
    })


geojson = {
    "type": "FeatureCollection",
    "features": features,
}

OUTPUT_GEOJSON.write_text(
    json.dumps(geojson),
    encoding="utf-8",
)


# -------------------------
# Reusable endpoint registry
# -------------------------
endpoint_registry.to_csv(
    ENDPOINT_REGISTRY_CSV,
    index=False,
)

endpoint_features = []

for _, row in endpoint_registry.iterrows():
    if not row["found"]:
        continue

    if pd.isna(row["longitude"]) or pd.isna(row["latitude"]):
        continue

    properties = {
        key: row.get(key)
        for key in [
            "endpoint_name",
            "normalized_endpoint",
            "state",
            "voltage_kv",
            "resolved_name",
            "source",
            "feature_id",
            "feature_type",
            "name_match",
            "matched_alias",
            "voltage_support",
            "operator",
            "operator_kind",
            "source_file",
        ]
    }

    for key, value in list(
        properties.items()
    ):
        if pd.isna(value):
            properties[key] = None

    endpoint_features.append({
        "type": "Feature",
        "properties": properties,
        "geometry": {
            "type": "Point",
            "coordinates": [
                float(row["longitude"]),
                float(row["latitude"]),
            ],
        },
    })


ENDPOINT_REGISTRY_GEOJSON.write_text(
    json.dumps({
        "type": "FeatureCollection",
        "features": endpoint_features,
    }),
    encoding="utf-8",
)


print("Saved:")
print(" -", OUTPUT_CSV)
print(" -", OUTPUT_GEOJSON)
print(" -", ENDPOINT_REGISTRY_CSV)
print(" -", ENDPOINT_REGISTRY_GEOJSON)
print()
print("Resolved/review route geometries:", len(features))
print("Resolved endpoint points:", len(endpoint_features))


## What changed in this refactor

The pathfinder is mostly the same. The endpoint resolver is not.

The old notebook could miss an obvious facility because:

- HIFLD substation points were not searched at all;
- OSM station voltage tags could hard-reject an otherwise exact facility name;
- only one OSM cache type was searched;
- shorthand aliases were incomplete;
- cross-border tie endpoints were too easy to miss;
- endpoint discoveries were not turned into a reusable registry.

The refactored notebook fixes those before touching the routing algorithm.

The most useful output for debugging is now:

```text
dominion_endpoint_registry.csv
```

Check that table first. If a named endpoint is present in public GIS under a capitalization change, minor typo, suffix like `Substation`, or one of the explicit aliases, it should be visible there before route reconstruction begins.
